(sec:multi_layer_neural_networks)=

# Multi-layer Neural Networks

So far, we have considered single-layer neural networks, or more precisely, neural networks with a single *hidden* layer. In the previous chapter, we saw that such models can already learn complex patterns, such as non-linear decision boundaries. In fact, it can be shown mathematically that single-layer neural networks are capable of *representing any continuous function to arbitrary accuracy*, provided that they have enough neurons. This result is known as the **universal approximation theorem**. In practice, however, neural networks are usually scaled by stacking multiple hidden layers, leading to multi-layer or **deep neural networks**.

:::{hint} Universal Approximation Theorem
:class: dropdown

One formal version of the universal approximation theorem states the following. Let $u$ be a continuous function on a closed and bounded input region, and let $\varepsilon > 0$. Then there exists a single-layer neural network of the form {ref}`eq:slp_output` such that

$$
|\hat{y} - u(\vec{x})| < \varepsilon
$$

for every input $\vec{x}$ in that region.

The classic proof of the theorem is elegant, but it uses some advanced functional analysis, which is beyond the scope of this course.

Importantly, the theorem is an existence result. It tells us that some number of neurons $n$ and some set of weights $\bm{W}$ and $\vec{a}$ can approximate the target function arbitrarily well, but it does not tell us how to find the optimal parameters during training.

:::

## Theoretical Foundations

### Fully-Connected Neural Networks

To understand the mathematical properties of deep neural networks, let us first revisit a single-layer neural network. {ref}`fig:single_layer_perceptron` illustrates the computation of the network output $\hat{y}$ in red. Compared to {ref}`fig:neuron`, the summation of weighted input features in blue and the application of the activation function are combined into a single layer of nodes in green.

:::{figure} /assets/figures/deep_learning/slp.svg
:label: fig:single_layer_perceptron
:align: center
:width: 300px
:alt: Computational graph of a single hidden layer
Computational graph of a single hidden layer, where the weighted sums and activation functions are combined into one layer of hidden nodes.
:::

Formally, the output of this single layer with $n$ hidden neurons can be written as the vector

$$
\vec{h} = \sigma(\bm{W}^\top \vec{x}) \in \mathbb{R}^n.
$$

From this abstract representation, the generalisation to a multi-layer network is straightforward: we add additional layers of nodes between the input and output, as illustrated in {ref}`fig:multi_layer_perceptron`.

:::{figure} /assets/figures/deep_learning/mlp.svg
:label: fig:multi_layer_perceptron
:align: center
:width: 500px
:alt: Computational graph of a multi-layer neural network
Computational graph of a multi-layer neural network, where each hidden layer takes the output of the previous layer as its input.
:::

This can be understood as each layer taking the output of the previous layer as its input:

$$
\vec{h}_l = \sigma_l(\bm{W}_l^\top \vec{h}_{l-1}) \in \mathbb{R}^{n_l},
$$

where we have again assumed an implicit node $(\vec{h}_l)_0 = 1$ in each layer to account for the bias term. Crucially, each layer can have its own number of neurons $n_l$, its own activation function $\sigma_l$, and independent weights $\bm{W}_l \in \mathbb{R}^{n_{l-1} \times n_l}$ that **fully connect** it with the previous layer. For a given feature vector $\vec{x}_i$, the output $\hat{\vec{y}}_i$ of the network is then given by the **composition** of all layers:

$$
\hat{\vec{y}}_i = \vec{h}_L \circ \vec{h}_{L-1} \circ \cdots \circ \vec{h}_2 \circ \vec{h}_1(\vec{x}_i)
$$

In contrast to previous examples, we assume that the output $\hat{\vec{y}}_i$ is a vector, which also constitutes the last layer $\vec{h}_L$ of the network.

:::{tip} Multiclass Classification with Neural Networks

Vector outputs $\hat{\vec{y}}$ can be used for multiclass classification, where each output neuron $y_j$ represents the probability that the input $\vec{x}$ belongs to class $j$. To guarantee that the output vector is a valid probability distribution, the activation function of the last layer is typically chosen to be the **softmax function**:

$$
\sigma_L(z) = \frac{\exp(z)}{\sum_{j=1}^{n_L} \exp(z_j)}
$$

This is usually combined with a **cross-entropy loss** function for training. For regression tasks, the output layer is typically a single neuron with an identity activation function, so the output is not constrained to a fixed range of values.

:::

### Other Variants of Neural Networks

The variety of deep neural network architectures is not limited to the fully connected networks introduced above. Many architectures are designed for data with a specific structure, such as images, sequences, text, or molecular graphs. The main idea is usually to build this structure into the network, either to reduce the number of parameters or to make learning easier.

**Convolutional Neural Networks (CNNs)** are widely used for image-like data, where neighbouring pixels or grid points are strongly related. Instead of connecting every neuron to every neuron in the previous layer, CNNs learn small filters that are applied across different spatial positions with shared weights. This reduces the number of parameters and allows the network to learn local patterns that can appear at different positions.

**Recurrent Neural Networks (RNNs)** are designed for sequential data, such as time series or text. They process one element of the sequence at a time and update a hidden state that carries information from previous steps. Because information is passed recursively through many steps, RNNs can suffer from vanishing or exploding gradients, which motivated variants such as Long Short-Term Memory (LSTM) networks.

**Transformer networks** are also used for sequential data, but they avoid processing the sequence strictly step by step. Instead, they use attention mechanisms that allow each element of a sequence to compare itself directly with all other elements. This makes it easier to capture long-range dependencies and allows much more parallel computation than classical RNNs. Transformers are now central in many language models and are increasingly used for chemical sequences and molecular representations.

**Graph Neural Networks (GNNs)** are designed for data represented as graphs, such as molecules, where atoms are nodes and bonds are edges. Rather than assuming a fixed grid or sequence, GNNs update each node representation by combining information from its neighbours. This makes them well suited for molecular property prediction, because the network can learn from both atom features and molecular connectivity. After several update steps, the node representations can be combined into a graph-level representation for classification or regression.

### Backpropagation

Training a deep neural network means finding the weights $\bm{W}_l$ for all layers $l = 1, \ldots, L$ that minimise a loss function $\mathcal{L}$ over a training dataset. This is usually done with known optimisation methods such as mini-batch gradient descent. To update the weights, we need the gradient of the loss function with respect to the weights in each layer. Because deep neural networks are built as compositions of layers, these gradients are obtained by repeated application of the chain rule. As a result, the gradient with respect to the weights in a given layer depends on the gradients of all subsequent layers, so we compute the gradients from the last layer back to the first. Algorithmically, this procedure is known as **backpropagation**.

:::{note} Derivation of gradients
:class: dropdown

For a single data point $(\vec{x}_i, \vec{y}_i)$, we write the forward pass as

$$
\vec{a}_l = \bm{W}_l^\top \vec{h}_{l-1}, \qquad \vec{h}_l = \sigma_l(\vec{a}_l),
$$

where $\vec{h}_0 = \vec{x}_i$ and the bias is already absorbed into the weight matrix by adding an implicit constant feature to each layer. To compute the gradients, we define the error vector

$$
\vec{\delta}_l := \nabla_{\vec{a}_l} \ell(\vec{h}_L, \vec{y}_i),
$$

which measures how strongly the loss changes with the pre-activation values in layer $l$. For the last layer, this is obtained directly from the loss and the activation function,

$$
\vec{\delta}_L = \nabla_{\vec{h}_L} \ell(\vec{h}_L, \vec{y}_i) \odot \sigma_L'(\vec{a}_L),
$$

where $\odot$ denotes element-wise multiplication. For earlier layers, repeated use of the chain rule gives the backward recursion

$$
\vec{\delta}_l = (\bm{W}_{l+1} \vec{\delta}_{l+1}) \odot \sigma_l'(\vec{a}_l), \qquad l = L-1, \ldots, 1.
$$

Once $\vec{\delta}_l$ is known, the gradient with respect to the whole weight matrix of layer $l$ is simply

$$
\nabla_{\bm{W}_l} \ell = \vec{h}_{l-1} \vec{\delta}_l^\top.
$$

Thus, after one forward pass has stored the activations $\vec{a}_l$ and $\vec{h}_l$, the gradients can be computed by moving backward from the last layer to the first. For a batch of data points, the corresponding gradients are computed for each point and averaged before the weights are updated.

:::

## Deep-Learning Frameworks

Computing gradients with backpropagation is one of the most time- and resource-consuming parts of training a deep neural network. Instead of deriving gradients and implementing backpropagation manually for every neural network, we can use deep-learning frameworks that provide efficient tools for building and training these models. Here, we briefly introduce [**PyTorch**](https://pytorch.org/), which provides several important features.

**Automatic differentiation** means that mathematical operations are internally represented as a computational graph. This graph is traversed backwards, and the chain rule is applied automatically to compute gradients, enabling efficient backpropagation without manually deriving each derivative.

**GPU acceleration** is useful because many neural-network computations reduce to large matrix multiplications. These operations can be parallelised efficiently on modern GPUs, which often makes training much faster than on a CPU.

**Rich ecosystem** refers to the large collection of tools and libraries provided by PyTorch, including object-based implementations of neural network layers, optimisers, loss functions, and dataloaders.

<!-- :::{tip} Automatic differentiation example
:class: dropdown

To illustrate automatic differentiation, consider the simple function

$$
y = e^{wx} + w^2,
$$

where $x$ is an input and $w$ is a parameter. Automatic differentiation first represents this calculation as a computational graph by introducing intermediate variables:

$$
z_1 = wx, \qquad z_2 = e^{z_1}, \qquad z_3 = w^2, \qquad y = z_2 + z_3.
$$

:::{figure} /assets/figures/deep_learning/forward_backward.svg
:label: fig:forward_backward
:align: center
:width: 300px
:alt: Forward and backward pass through a computational graph
Computational graph of $y = e^{wx} + w^2$, illustrating how a forward pass stores intermediate values and a backward pass propagates gradients.
:::

For a given pair of values $(x,w) = (1,2)$, the **forward pass** evaluates these nodes in order and stores their values:

$$
z_1 = wx = 2, \qquad z_2 = e^{z_1} = e^2, \qquad z_3 = w^2 = 4, \qquad y = z_2 + z_3 = e^2 + 4.
$$

The **backward pass** starts from $\bar{y}=1$, because $\partial y / \partial y = 1$, and then propagates sensitivities backward through the graph using the chain rule. For example, the contribution through $z_2=e^{z_1}$ gives $\bar{z}_1 = e^{z_1}\bar{z}_2$, while the contribution through $z_1=wx$ gives contributions to both $w$ and $x$.

In general, each node collects contributions from all later nodes that depend on it:

$$
\bar{v}_i = \sum_{v_j \in \mathrm{children}(v_i)} \frac{\partial v_j}{\partial v_i}\bar{v}_j.
$$

Applying this to the graph above gives

$$
\frac{\partial y}{\partial w} = x e^{wx} + 2w, \qquad \frac{\partial y}{\partial x} = w e^{wx}.
$$

::: -->

Below, we demonstrate how to use PyTorch's ecosystem to train a three-layer neural network for classifying the two concentric circles from the previous chapter. In the next section, we will move on to a problem from chemistry.

In [8]:
import torch
from torch import nn
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# Generate data
X, y = make_circles(n_samples=200, factor=0.3, noise=0.08, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert Numpy arrays to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define model
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# Define loss function and optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
n_epochs = 500
for _ in range(n_epochs):
    logits = model(X_train)         # forward pass
    loss = loss_fn(logits, y_train) # compute loss

    optimizer.zero_grad()           # reset old gradients to zero
    loss.backward()                 # compute gradients
    optimizer.step()                # update parameters

# Evaluation on test set
with torch.no_grad():               # disable gradient computation
    test_logits = model(X_test)
    y_pred = (torch.sigmoid(test_logits) >= 0.5).float()
    test_accuracy = (y_pred == y_test).float().mean()

print(f"Test accuracy: {test_accuracy.item():.2f}")

Test accuracy: 1.00


## Self-Study Questions

1. Explain the statement from the universal approximation theorem in your own words.
2. Why does backpropagation compute gradients from the last layer back to the first layer?
3. Try out modifying the neural-network architecture from the example above by changing the number of layers, neurons, or activation functions. 